# 02 — Contract Chunking Strategy Comparison

Evaluates three chunking configurations to find the optimal chunk size for procurement contract retrieval.

**Configurations tested:**

| Config | Chunk Size | Overlap | Verdict |
|--------|-----------|---------|----------|
| A | 512 chars | 64 | Too granular — splits clauses mid-sentence |
| B | 800 chars | 150 | ✅ **Selected** — preserves full clauses |
| C | 1200 chars | 200 | Too broad — dilutes retrieval precision |

**Evaluation method:** Manually assess chunk quality against 5 procurement-specific retrieval questions.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from dotenv import load_dotenv
load_dotenv('../.env')

from src.ingestion.pdf_loader import load_documents
from src.ingestion.chunker    import RecursiveChunker, chunk_documents, chunk_stats

plt.style.use('seaborn-v0_8-whitegrid')
print('Setup complete')

## 1. Load Contract Documents

In [ ]:
docs = load_documents('../data/contracts', recursive=True)

print(f'Documents loaded: {len(docs)}')
print(f'{"─"*50}')
for doc in docs:
    total_chars = sum(len(p.text) for p in doc.pages)
    print(f'  [{doc.doc_type}] {doc.source_file}')
    print(f'    pages: {doc.total_pages}  |  chars: {total_chars:,}')

## 2. Run Three Chunking Configurations

In [ ]:
configs = [
    {'name': 'A — Small',   'chunk_size': 512,  'overlap': 64,  'color': '#dc3545'},
    {'name': 'B — Medium',  'chunk_size': 800,  'overlap': 150, 'color': '#28a745'},
    {'name': 'C — Large',   'chunk_size': 1200, 'overlap': 200, 'color': '#fd7e14'},
]

results = {}
for cfg in configs:
    chunks = chunk_documents(docs, chunk_size=cfg['chunk_size'], overlap=cfg['overlap'])
    lengths = [len(c.text) for c in chunks]
    results[cfg['name']] = {
        'chunks'   : chunks,
        'lengths'  : lengths,
        'count'    : len(chunks),
        'avg'      : int(np.mean(lengths)),
        'min'      : min(lengths),
        'max'      : max(lengths),
        'config'   : cfg,
    }
    print(f"Config {cfg['name']}: {len(chunks)} chunks | avg {int(np.mean(lengths))} chars")

## 3. Chunk Size Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
fig.suptitle('Chunk Length Distributions by Configuration', fontsize=13, fontweight='bold')

for ax, (name, data) in zip(axes, results.items()):
    cfg    = data['config']
    color  = cfg['color']
    lengths= data['lengths']

    ax.hist(lengths, bins=25, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(np.mean(lengths), color='black', linestyle='--', linewidth=1.5,
               label=f'Mean: {int(np.mean(lengths))}')
    ax.set_title(f'{name}\n(size={cfg["chunk_size"]}, overlap={cfg["overlap"]})')
    ax.set_xlabel('Chunk length (chars)')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=9)
    ax.text(0.97, 0.95, f'n={len(lengths)}', transform=ax.transAxes,
            ha='right', va='top', fontsize=9, color='#555')

plt.tight_layout()
plt.savefig('../data/chunking_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Chunk Count vs Coverage Trade-off

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

names  = list(results.keys())
counts = [r['count'] for r in results.values()]
avgs   = [r['avg']   for r in results.values()]
colors = [r['config']['color'] for r in results.values()]

x = np.arange(len(names))
bars = ax.bar(x, counts, color=colors, width=0.5, edgecolor='white')

for bar, count, avg in zip(bars, counts, avgs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{count} chunks\navg {avg}c', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_title('Total Chunk Count by Configuration', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Chunks')
ax.set_ylim(0, max(counts) * 1.25)

# Annotate the winner
ax.annotate('✅ Selected', xy=(1, counts[1]),
            xytext=(1.35, counts[1] * 1.1),
            arrowprops=dict(arrowstyle='->', color='#28a745'),
            color='#28a745', fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Qualitative Clause Preservation Test

In [ ]:
# Test how well each config preserves key contract clauses
# by checking if important phrases appear intact within single chunks

KEY_PHRASES = [
    'late delivery penalty',
    'payment terms',
    'termination for convenience',
    'force majeure',
    'single-source',
]

print('CLAUSE PRESERVATION — phrases found intact in single chunks')
print(f'{"─"*65}')
print(f'{"":<25} {"Config A":>12} {"Config B":>12} {"Config C":>12}')
print(f'{"─"*65}')

for phrase in KEY_PHRASES:
    row = [f'  {phrase:<23}']
    for name, data in results.items():
        matches = sum(1 for c in data['chunks'] if phrase.lower() in c.text.lower())
        row.append(f'{matches:>12}')
    print(''.join(row))

print(f'{"─"*65}')
print('Higher = phrase appears in more chunks (better retrieval coverage)')

## 6. Sample Chunks — Side by Side Comparison

In [ ]:
# Find a chunk containing 'penalty' in each config and compare
SEARCH_TERM = 'penalty'

print(f'SAMPLE CHUNKS containing "{SEARCH_TERM}" — one per config')

for name, data in results.items():
    cfg = data['config']
    matching = [c for c in data['chunks'] if SEARCH_TERM.lower() in c.text.lower()]

    print(f'\n{"═"*65}')
    print(f'Config {name} (size={cfg["chunk_size"]}, overlap={cfg["overlap"]})')
    print(f'Matching chunks: {len(matching)}')
    print(f'{'─'*65}')

    if matching:
        sample = matching[0]
        print(f'Source : {sample.source_file} | page {sample.page_number}')
        print(f'Length : {len(sample.text)} chars')
        print(f'Text   :')
        print(f'  {sample.text[:400]}...' if len(sample.text) > 400 else f'  {sample.text}')
    else:
        print('  (no matching chunks — term may have been split across chunk boundary)')

## 7. Simulated Retrieval Quality (Pre-RAGAS)

In [ ]:
# Simulate what RAGAS would measure by scoring:
# - How many test questions have at least one highly relevant chunk?
# - How much noise (irrelevant content) is in the top-5 retrieved chunks?

TEST_QUESTIONS = [
    ('penalty clause',        ['penalty', 'late delivery', '0.5%', 'delay']),
    ('payment terms',         ['payment', 'invoice', 'days', 'net']),
    ('termination notice',    ['termination', 'notice', 'convenience', 'days']),
    ('force majeure',         ['force majeure', 'beyond', 'control', 'event']),
    ('single-source policy',  ['single-source', 'sole source', '100,000', 'approval']),
]

def score_chunks_for_question(chunks, keywords, top_k=5):
    """Score chunks by keyword overlap (proxy for retrieval quality)."""
    scored = []
    for chunk in chunks:
        text_lower = chunk.text.lower()
        hits = sum(1 for kw in keywords if kw.lower() in text_lower)
        scored.append((chunk, hits))
    scored.sort(key=lambda x: x[1], reverse=True)
    top = scored[:top_k]
    relevant = sum(1 for _, hits in top if hits > 0)
    return relevant / top_k  # precision@k

print('SIMULATED RETRIEVAL PRECISION@5 BY CONFIG')
print(f'{"─"*60}')
print(f'{"Question":<35} {"Config A":>9} {"Config B":>9} {"Config C":>9}')
print(f'{"─"*60}')

config_totals = {name: 0.0 for name in results.keys()}

for question, keywords in TEST_QUESTIONS:
    row = [f'  {question:<33}']
    for name, data in results.items():
        score = score_chunks_for_question(data['chunks'], keywords)
        config_totals[name] += score
        row.append(f'{score:>9.2f}')
    print(''.join(row))

print(f'{"─"*60}')
avg_row = [f'  {"AVERAGE":<33}']
for name in results.keys():
    avg = config_totals[name] / len(TEST_QUESTIONS)
    avg_row.append(f'{avg:>9.2f}')
print(''.join(avg_row))
print(f'{"─"*60}')

In [ ]:
# Visualise precision scores
fig, ax = plt.subplots(figsize=(10, 4))

questions = [q for q, _ in TEST_QUESTIONS]
x         = np.arange(len(questions))
width     = 0.25

for i, (name, data) in enumerate(results.items()):
    scores = [
        score_chunks_for_question(data['chunks'], kws)
        for _, kws in TEST_QUESTIONS
    ]
    ax.bar(x + i * width, scores, width, label=name,
           color=data['config']['color'], edgecolor='white')

ax.set_xticks(x + width)
ax.set_xticklabels(questions, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('Precision@5')
ax.set_ylim(0, 1.15)
ax.set_title('Simulated Retrieval Precision@5 by Chunking Config', fontsize=12, fontweight='bold')
ax.legend()
ax.axhline(0.8, color='gray', linestyle='--', alpha=0.5, label='Target 0.80')

plt.tight_layout()
plt.savefig('../data/chunking_precision.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Decision

### Selected: Config B — chunk_size=800, overlap=150

**Reasoning:**

| Criterion | Config A (512) | Config B (800) ✅ | Config C (1200) |
|-----------|---------------|-----------------|----------------|
| Clause preservation | ❌ Often splits clauses mid-sentence | ✅ Full clauses fit within one chunk | ⚠ Multiple clauses merged, diluting precision |
| Retrieval precision | ⚠ High chunk count, more noise | ✅ Focused, relevant chunks | ⚠ Broad chunks contain off-topic content |
| Context window fit | ✅ Small, many fit in LLM context | ✅ 5 chunks ≈ 4000 chars — fits comfortably | ⚠ 5 chunks ≈ 6000 chars — approaching limit |
| ChromaDB storage | ⚠ Most chunks = highest storage | ✅ Balanced | ✅ Fewest chunks |

**Key insight:** Procurement contracts have well-defined clauses (payment, penalty, termination) that are typically 200–600 characters. Config B's 800-char chunks reliably capture complete clauses including their header and sub-points, while 512 chars often splits them.

These settings are coded into `src/ingestion/chunker.py` as defaults.